In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.1 Graphs, Incidence Matrices, and the Laplacian

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter VI — Structure, Graphs, and Fast Algorithms",
    number="6.1",
    title="Graphs, Incidence Matrices, and the Laplacian",
    blurb="A graph is a matrix wearing a disguise: the incidence matrix "
    "turns edges into rows, its Gram matrix is the Laplacian, the null "
    "space counts connected components, the second eigenvector cuts "
    "clusters, a cofactor counts spanning trees — and Kirchhoff's laws turn "
    "out to be the four fundamental subspaces doing physics.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

Chapter V computed with matrices whose structure came from a grid. Chapter VI
widens the lens: structure of *any* shape, starting with the most general
discrete structure there is — a graph. The dictionary is exact. Build the
**incidence matrix** $B$ (one row per edge, $-1$ and $+1$ marking its
ends) and the graph's whole anatomy becomes linear algebra: the **Laplacian**
$L = B^{\top}B = D - A$ (an equality of integer matrices, gated *exactly*),
its null space spanned by the indicator vectors of connected components,
its second eigenvalue — Fiedler's algebraic connectivity
{cite}`fiedler1973` — measuring how hard the graph is to cut, and the sign
pattern of the matching eigenvector *performing* the cut: spectral
bisection recovers a planted two-cluster structure exactly, with a 29-fold
spectral gap certifying the verdict.

The closing exercises are the classical theorems, checked against brute
force and physics. The **matrix-tree theorem** says one cofactor of $L$
counts spanning trees; this notebook counts all 24 of the worked graph's
trees by exhaustive enumeration and matches the determinant integer for
integer. And the electrical reading — edges as unit resistors — makes
$L^{+}$ compute **effective resistance**, verified against the series and
parallel rules every physics course teaches, plus Foster's theorem
(the resistances over edges sum to exactly $n - 1$), plus Kirchhoff's two
laws landing on the four fundamental subspaces of $B$: currents balance
because $B^{\top}$ says so, voltages close around loops because the cycle
space is $B^{\top}$'s null space. [§1.4](../01-matrices/four-subspaces.ipynb)'s
abstract picture, with electrons in it.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Strang {cite}`strang2023` Section 3.5 for incidence matrices
> and the four-subspace reading; von Luxburg {cite}`vonluxburg2007` for
> spectral clustering; Fiedler's original {cite}`fiedler1973`. Everything
> here is exact or eigvalsh-grade — the numerics are Chapter V's, at rest.

## Theory in brief

### The incidence matrix and the Laplacian

Orient each of the $m$ edges arbitrarily; the incidence matrix
$B \in \mathbb{R}^{m\times n}$ has $B_{e,i} = -1$, $B_{e,j} = +1$ for edge
$e = (i, j)$. Then, with $D$ the diagonal of degrees and $A$ the adjacency
matrix,

```{math}
:label: eq-gl-laplacian
L \;=\; B^{\top}B \;=\; D - A, \qquad
x^{\top}Lx \;=\; \sum_{(i,j)\in E}(x_i - x_j)^2 \;\ge\; 0 ,
```

an identity of integer matrices (the orientation cancels in the Gram
product) and an explicit certificate that $L$ is positive semidefinite.
The quadratic form says everything: $Lx = 0$ iff $x$ is constant on every
connected component, so

```{math}
:label: eq-gl-nullspace
\dim \operatorname{null}(L) \;=\; \#\{\text{connected components}\},
```

and the eigenvalue $0$ has the component indicators as its eigenspace.

### Fiedler's eigenvalue, and cutting graphs

For a connected graph the second-smallest eigenvalue $\lambda_2 > 0$ — the
**algebraic connectivity** — and its eigenvector $v_2$ minimizes the edge
quadratic $\sum (x_i - x_j)^2$ among unit vectors orthogonal to the
constant. A graph that is two dense clusters joined by a few edges can make
this quadratic tiny by putting one cluster at $+$ and the other at $-$:
the sign pattern of $v_2$ *is* the cut. On a path graph the same
minimization gives the smoothest non-constant profile — a discrete cosine,
$v_2(i) \propto \cos\bigl(\pi(i + \tfrac12)/n\bigr)$, with eigenvalues
$2 - 2\cos(k\pi/n)$: [§5.3](../05-numerical/sparse-matrices.ipynb)'s
Poisson spectrum, rediscovered as graph theory.

### Trees and resistors

Two classical theorems close the loop. The **matrix-tree theorem**: delete
any one row and column of $L$; the determinant of what remains counts the
spanning trees. And the **electrical network**: edges as unit resistors,
a current injected at $i$ and drawn at $j$, potentials
$x = L^{+}(e_i - e_j)$, and

```{math}
:label: eq-gl-reff
R_{\text{eff}}(i,j) \;=\; (e_i - e_j)^{\top}L^{+}(e_i - e_j),
\qquad \sum_{(i,j)\in E} R_{\text{eff}}(i,j) \;=\; n - 1
```

(Foster's theorem — the sum over *edges* is an integer, whatever the
graph). Kirchhoff's current law is $B^{\top}y = f$ (flows balance at
nodes); the voltage law is $y \perp \operatorname{null}(B^{\top})$
(drops cancel around cycles) — the four subspaces of
[§1.4](../01-matrices/four-subspaces.ipynb), carrying current.

---
## Setup

Data and instruments only: the worked graph's edges and layout, and a
matplotlib graph-drawing helper. The incidence matrix and Laplacian —
the notebook's dictionary — are built in Exercise 1.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt

from ecp import validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps

# The worked graph: 8 nodes, 10 edges, drawn with these hand-laid positions.
# Two triangles sharing structure on the left, a diamond on the right, one
# pendant node — enough anatomy for every theorem below.
EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3),
         (3, 4), (4, 5), (4, 6), (5, 6), (6, 7)]
N_NODES = 8
POS = np.array([[0.0, 1.0], [1.0, 1.7], [1.0, 0.3], [2.0, 1.0],
                [3.0, 1.0], [4.0, 1.7], [4.0, 0.3], [5.0, 1.0]])




# instrument: matplotlib plumbing — nodes, edges, labels; no linear algebra
# inside, and no exercise's lesson.
def draw_graph(ax, edges, pos, colors=None, labels=True):
    """Draw a small graph with matplotlib primitives (no graph library)."""
    for i, j in edges:
        ax.plot(*zip(pos[i], pos[j]), color="#9aa3b8", lw=1.3, zorder=1)
    c = colors if colors is not None else ["#16213e"] * len(pos)
    ax.scatter(pos[:, 0], pos[:, 1], s=340, c=c, zorder=2,
               edgecolors="#16213e", linewidths=1.2)
    if labels:
        for k, (x, y) in enumerate(pos):
            ax.annotate(str(k), (x, y), ha="center", va="center",
                        fontsize=10, color="white", zorder=3)
    ax.set_aspect("equal")
    ax.axis("off")

## Exercise 1: The dictionary: $L = B^{\top}B = D - A$, exactly

{eq}`eq-gl-laplacian` is an identity of integer matrices, and integer
arithmetic below $2^{53}$ is exact in floating point — so for once the
course's no-exact-comparison rule steps aside, deliberately.

**Part a)** Write `incidence(edges, n)` — one row per edge, $-1$ at the
tail, $+1$ at the head — and with it build the incidence matrix of the worked
graph and print it: 10 rows (edges), 8 columns (nodes), each row one $-1$
and one $+1$.

**Part b)** Form $L = B^{\top}B$, the degree matrix $D$ (diagonal of row
sums of the adjacency matrix), and $A$ itself. Gate
`np.array_equal(L, D - A)` — **exactly**, because every entry is a small
integer and the Gram product cancels the arbitrary edge orientations.

**Part c)** Certify positive semidefiniteness twice: by the identity
$x^{\top}Lx = \sum_{(i,j)}(x_i - x_j)^2$ evaluated on 100 random vectors
(never negative), and by `np.linalg.eigvalsh` ($\lambda_{\min}$ within
$100\,\varepsilon\lVert L\rVert$ of zero — the constant vector is in the
null space, so $0$ *is* an eigenvalue).

**Part d)** Draw the graph beside `la.sparsity_pattern` of $L$: the
pattern is the adjacency structure plus the diagonal, which is the entire
point — the Laplacian *is* the graph.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    dictionary_exact,
    "L = B'B = D - A as an exact identity of integer matrices (Eq. 1)",
    "small integers are exactly representable and the Gram product cancels "
    "the arbitrary orientations — the legitimate exact comparison",
)
validate.check(
    quad_gap < 1e-12 and bool(np.all(edge_form >= 0.0)),
    "and x'Lx is the sum of squared edge differences: psd by identity",
    "the quadratic form certificate on 100 random vectors — nonnegativity "
    "needs no eigenvalue computation at all",
)
validate.below(
    abs(ev_L[0]), 100 * EPS * norm_L,
    "with lambda_1 = 0 to eigvalsh accuracy",
    "the constant vector is in the null space by Eq. 1; the tolerance is "
    "scaled by the matrix, per the course's rules",
)

## Exercise 2: The null space counts components

{eq}`eq-gl-nullspace` turns a topology question into a rank computation.

**Part a)** For the (connected) worked graph, confirm the zero eigenvalue
is simple: exactly one eigenvalue below $10^{-8}$, the rest above the
algebraic connectivity $\lambda_2 = 0.24$ — the counting is robust because
the spectral gap is $O(1)$, not because the arithmetic is exact.

**Part b)** Now break it deliberately: delete edges $(3,4)$ and $(6,7)$,
leaving three components $\{0,1,2,3\}$, $\{4,5,6\}$, $\{7\}$. Gate
exactly three eigenvalues below $10^{-8}$, with the fourth at $O(1)$.

**Part c)** Confirm the eigenspace is the indicator space: for each
component $C$, the indicator vector $\mathbf{1}_C$ satisfies
$\lVert L\,\mathbf{1}_C\rVert = 0$ *exactly* (integer arithmetic — each row
of $L\mathbf{1}_C$ sums integers that cancel).

**Part d)** Confirm the rank bookkeeping: `np.linalg.matrix_rank` of the
broken graph's $L$ is $n - 3 = 5$, and of its incidence matrix likewise —
rank is the number of nodes minus the number of components, for any graph.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    n_zero_conn == 1 and n_zero_cut == 3,
    "the multiplicity of eigenvalue zero counts connected components (Eq. 2)",
    "1 for the connected graph, 3 after two edges are cut — the count is "
    "robust because the spectral gap to the first nonzero eigenvalue is "
    "O(1), a property of the graph, not of the arithmetic",
)
validate.check(
    indicator_exact,
    "with the component indicators exactly in the null space",
    "each row of L times an indicator sums integers that cancel — exact "
    "zero, legitimately",
)
validate.check(
    rank_L == 5 and rank_B == 5,
    "and rank(L) = rank(B) = n minus the component count",
    "the null space of Eq. 2 seen from the rank side, for both the "
    "Laplacian and the incidence matrix",
)

## Exercise 3: A zoo of spectra, and the Poisson matrix in disguise

Whole families of graphs have closed-form spectra, and one of them is an
old friend.

**Part a)** Build Laplacians for the path $P_{30}$, the cycle $C_{30}$,
and the complete graph $K_{30}$. Confirm the closed forms with
`np.linalg.eigvalsh`, each sorted against its prediction:

- path: $\lambda_k = 2 - 2\cos(k\pi/30)$, $k = 0,\dots,29$ — gate to
  $10^{-12}$;
- cycle: $\lambda_k = 2 - 2\cos(2\pi k/30)$ — gate to $10^{-12}$;
- complete: $\{0\}\cup\{30\text{ (29-fold)}\}$ — gate to $10^{-12}$.

**Part b)** The path Laplacian *is* [§5.3](../05-numerical/sparse-matrices.ipynb)'s
1-D Poisson matrix with free (Neumann) ends — the grid was a graph all
along. Its second eigenvector is the smoothest non-constant profile:
gate $|v_2^{\top}c| > 1 - 10^{-10}$ against the normalized discrete cosine
$c_i = \cos\bigl(\pi(i+\tfrac12)/30\bigr)$.

**Part c)** Build the zoo's Laplacians and report their spectra; the four
of them (path, cycle, complete, and Exercise 4's clustered graph) are drawn
as columns of dots in Exercise 4's combined figure, beside the graph they
calibrate.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    gap_path < 1e-12 and gap_cyc < 1e-12 and gap_comp < 1e-12,
    "path, cycle and complete-graph spectra match their closed forms",
    f"gaps {gap_path:.0e}, {gap_cyc:.0e}, {gap_comp:.0e}: "
    "2 - 2cos(k pi/n), 2 - 2cos(2 pi k/n), and {0, n} — three families, "
    "one eigvalsh",
)
validate.check(
    align > 1.0 - 1e-10,
    "and the path's Fiedler vector is the discrete cosine",
    "the smoothest non-constant profile — 5.3's Poisson eigenvector, "
    "rediscovered as graph theory",
)

## Exercise 4: Spectral bisection: the sign pattern is the cut

The planted-cluster test, fully deterministic: two complete graphs
$K_{10}$ joined by exactly two bridge edges $(0, 15)$ and $(2, 17)$.

**Part a)** Build the graph and its Laplacian ($n = 20$,
$m = 2\binom{10}{2} + 2 = 92$ edges). Compute the spectrum: the Fiedler
value $\lambda_2 = 0.343$ sits far below $\lambda_3 = 10.0$ — a 29-fold
gap, which is the certificate that a *single* cut direction dominates.

**Part b)** Gate the bisection: the sign pattern of $v_2$ puts nodes
$0\!-\!9$ on one side and $10\!-\!19$ on the other, **exactly** — and the
smallest component magnitude is $0.19$, so the signs are nowhere near a
rounding decision (that margin, not luck, is what makes an exact gate
legitimate here).

**Part c)** Confirm the cut it found is the planted one by counting: the
edges crossing the sign boundary are exactly the 2 bridges, against 90
intra-cluster edges.

**Part d)** Draw the graph with nodes coloured by $\operatorname{sign}(v_2)$
— two rings, two colours, two bridges — and the clustered spectrum joins
the zoo figure: one near-zero straggler ($\lambda_2$) under a bulk at
$\lambda \approx 10$, the picture of "almost two components".

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    ev_clu[2] / ev_clu[1] > 20.0,
    "the Fiedler value sits far below the bulk: one cut direction dominates",
    f"lambda_2 = {ev_clu[1]:.3f} against lambda_3 = {ev_clu[2]:.1f} — the "
    "29x gap is the certificate that bisection is answering a well-posed "
    "question",
)
validate.check(
    split_exact and min_comp > 0.1,
    "spectral bisection recovers the planted clusters exactly",
    f"all twenty signs correct, smallest component {min_comp:.2f}: the "
    "exact gate is legitimate because the margin is O(1), not because "
    "signs are always safe",
)
validate.check(
    crossing == 2,
    "and the cut it found is the planted one: exactly the two bridges cross",
    "2 crossing edges out of 92 — the sign pattern read back as a cut, "
    "counted exactly",
)

## Exercise 5: The matrix-tree theorem, against brute force

Kirchhoff's 1847 theorem: any cofactor of $L$ counts the spanning trees.
For the worked graph the claim is checkable by exhaustive enumeration —
which is exactly what this exercise does.

**Part a)** Compute the cofactor: delete row and column 0 of $L$ and take
`np.linalg.det` of the $7\times7$ remainder. The determinant of an integer
matrix is an integer; round it (the float error, about $10^{-14}$, is ten
orders below the spacing of integers).

**Part b)** Write `is_spanning_tree(subset)`: a union–find over the 8
nodes that returns whether 7 given edges connect everything without a
cycle.

**Write this one yourself** — union–find is three small functions, and it
is the certificate the theorem is checked against.

**Part c)** Enumerate all $\binom{10}{7} = 120$ edge subsets of size 7,
count the spanning trees, and gate the **exact integer equality** with the
cofactor: 24 = 24.

**Part d)** Confirm the theorem's invariance: the cofactor is the same
(24) whichever row/column $i$ is deleted — all 8 deletions agree after
rounding.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    cof == brute,
    "the matrix-tree cofactor equals the brute-force spanning-tree count",
    f"{cof} = {brute}, integer against integer: a determinant computed in "
    "floats, rounded across a 1e-14 error to a count certified by "
    "union-find enumeration",
)
validate.check(
    all(c == cof for c in cofs),
    "and every cofactor agrees — the theorem's stated invariance",
    "deleting any row and column gives 24: the choice of grounded node is "
    "arbitrary, as the electrical reading of Exercise 6 explains",
)

## Exercise 6: Resistor networks: $L^{+}$, physics, and the four subspaces

Read every edge as a 1-ohm resistor. Linear algebra becomes circuit
theory, and the classical rules become checks on {eq}`eq-gl-reff`.

**Part a)** Effective resistance via the pseudoinverse
([§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb)):
$R_{\text{eff}}(i,j) = (e_i - e_j)^{\top}L^{+}(e_i - e_j)$. Gate the two
closed forms every physics course teaches, on clean graphs: the ends of
the path $P_{10}$ give $R = 9$ (nine resistors in series), and adjacent
nodes of the cycle $C_{10}$ give $R = 9/10$ (1 Ω in parallel with 9 Ω) —
both to $10^{-10}$.

**Part b)** Foster's theorem on the worked graph: the effective
resistances summed over its ten *edges* equal exactly $n - 1 = 7$ — gate
to $10^{-10}$. (An integer from a pseudoinverse: the theorem is doing the
arithmetic.)

**Part c)** Kirchhoff's current law as the row space at work: inject one
ampere at node 0, extract at node 7, solve $x = L^{+}(e_0 - e_7)$ for the
potentials, get edge currents $y = Bx$. Gate $B^{\top}y = e_0 - e_7$ to
$10^{-12}$ — the residual is pseudoinverse-grade rounding (measured
$2\times10^{-14}$), and the gate keeps a 40x margin against another BLAS's
path per Rule 8. Current balances at every node, with the net one ampere
appearing only at the terminals.

**Part d)** Kirchhoff's voltage law as the null space of $B^{\top}$: the
cycle space has dimension $m - n + 1 = 3$ (gate via
`np.linalg.matrix_rank`), and a basis $Z$ of it (from the SVD's trailing
right singular vectors of $B^{\top}$) satisfies $Z^{\top}y = 0$ to
$10^{-13}$: potential drops cancel around every loop *because* $y = Bx$
lives in the column space of $B$, which is orthogonal to the cycle space.
[§1.4](../01-matrices/four-subspaces.ipynb), carrying current.

In [ ]:
# (solution hidden on the public site)


```{admonition} With your assistant
:class: tip
Effective resistance is a *distance* on the graph — and a better one than
hop count for cluster structure. Ask your assistant for
`resistance_matrix(L)` returning all pairwise $R_{\text{eff}}$, then check
it against the mathematics rather than a demo: (i) on the path graph,
$R_{\text{eff}}(i,j) = |i - j|$ exactly (series law, all pairs); (ii) it
is a metric — verify the triangle inequality over every triple of the
worked graph; (iii) on Exercise 4's planted graph, the largest
intra-cluster resistance is smaller than the smallest inter-cluster one
(compute both, state the margin). The check is yours.
```

### Validation 6

In [ ]:
validate.check(
    abs(r_series - 9.0) < 1e-10 and abs(r_parallel - 0.9) < 1e-10,
    "the pseudoinverse reproduces the series and parallel resistor rules",
    f"path ends {abs(r_series - 9.0):.1e}-close to 9, cycle neighbours "
    f"{abs(r_parallel - 0.9):.1e}-close to 9/10: "
    "Eq. 3 against the physics classroom",
)
validate.check(
    abs(foster - (N_NODES - 1)) < 1e-10,
    "Foster's theorem: edge resistances sum to exactly n - 1",
    f"{foster:.12f} against 7 — an integer emerging from ten pseudoinverse "
    "quadratic forms, because the trace of L+ L is the rank",
)
validate.below(
    kcl_res, 1e-12,
    "Kirchhoff's current law holds at every node",
    "B'y = f: flows balance except the one ampere at the terminals — the "
    "row space of B doing physics",
)
validate.check(
    cycle_dim == 3 and kvl_res < 1e-13,
    "and the voltage law is the cycle space: drops cancel around all 3 loops",
    f"null(B') has dimension m - n + 1 = {cycle_dim} and Z'y = 0 to "
    f"{kvl_res:.0e}: y = Bx lies in col(B), orthogonal to the cycle space — "
    "the four subspaces of 1.4, carrying current",
)

---
## Notebook summary

**The dictionary is exact.** $L = B^{\top}B = D - A$ gated as an identity
of integer matrices; positive semidefiniteness certified twice (the
edge-difference quadratic on 100 random vectors, and
$\lambda_1 = -2\times10^{-16}$); the null space counted components —
1 connected, exactly 3 after cutting two edges, with the component
indicators annihilated *exactly* and the rank landing on $n - c$ both for
$L$ and for $B$.

**Spectra come in closed forms, and one is an old friend.** Path, cycle
and complete-graph spectra matched $2 - 2\cos(k\pi/n)$,
$2 - 2\cos(2\pi k/n)$ and $\{0, n\}$ to $10^{-14}$; the path's Fiedler
vector aligned with the discrete cosine to $1 - 10^{-12}$ — the 1-D
Poisson matrix of [§5.3](../05-numerical/sparse-matrices.ipynb) was a graph
Laplacian all along.

**The sign pattern is the cut, when the gap certifies it.** On two
$K_{10}$s joined by two bridges, $\lambda_2 = 0.343$ sat 29-fold below
$\lambda_3$; the Fiedler signs recovered the planted split exactly with a
smallest component of $0.19$ — the exact gate justified by an $O(1)$
margin, not by optimism — and the cut crossed exactly the 2 planted
bridges out of 92 edges.

**The classical theorems survive brute force.** The matrix-tree cofactor
said 24; union–find enumeration over all 120 edge subsets said 24; all
eight cofactors agree. The pseudoinverse reproduced the series (9) and
parallel (9/10) resistor rules to $10^{-12}$, Foster's sum landed on the
integer 7, Kirchhoff's current law held at rounding level
($2\times10^{-14}$), and the voltage law
turned out to be a statement about $\operatorname{null}(B^{\top})$ — three
loops, orthogonal to every realizable current, exactly as
[§1.4](../01-matrices/four-subspaces.ipynb) promised in the abstract.

**Methods introduced.** `incidence`/`laplacian` constructors, quadratic-form
psd certificates, null-space component counting, the closed-form spectrum
zoo, Fiedler vectors and spectral bisection with gap certificates,
matrix-tree cofactors against union–find enumeration, effective resistance
via `np.linalg.pinv`, Foster's theorem, and Kirchhoff's laws as subspace
statements.

## Outlook

- **From bisection to clustering.** Normalized Laplacians, $k$-means on
  the first $k$ eigenvectors, and the conductance guarantees of Cheeger's
  inequality are the industrial extension of Exercise 4
  {cite}`vonluxburg2007` — [§6.5](kernels-gram-matrix.ipynb) meets them
  again from the kernel side.
- **Random walks live here too.** $D^{-1}A$ is a Markov transition matrix,
  and its spectral gap controls mixing —
  [§6.2](markov-perron-pagerank.ipynb) makes that the whole story, with
  PageRank as the guest star.
- **The circulant shortcut.** The cycle's cosine spectrum is no accident:
  every circulant matrix is diagonalized by the DFT, which is
  [§6.3](circulant-toeplitz-fft.ipynb)'s fast algorithm.
- **Laplacians at scale.** Spectral graph theory's production face is
  [§5.5](../05-numerical/krylov-gmres-preconditioning.ipynb)'s machinery:
  Lanczos for the Fiedler pair, multigrid-preconditioned CG for
  $L x = f$ — the two chapters meeting in the middle.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()